# Control Sintético: Efecto de la Proposición 99 de California

## Basado en Abadie, Diamond y Hainmueller (2010)

**Paper original:** [Synthetic Control Methods for Comparative Case Studies](https://www.nber.org/system/files/working_papers/t0335/t0335.pdf)

---

### Contexto del problema

En 1988, California aprobó la *Tobacco Tax and Health Protection Act* (Proposición 99), que impuso un impuesto estatal de **25 centavos por paquete** sobre la venta de cigarrillos. Los ingresos generados se destinaron a programas de salud, medio ambiente y campañas anti-tabaco.

**Pregunta causal:** ¿Cuál fue el efecto de la Proposición 99 sobre las ventas per cápita de cigarrillos en California?

### ¿Por qué no basta con un simple antes/después?

Si simplemente comparamos las ventas en California antes y después de 1988, confundimos el efecto de la política con tendencias generales (las ventas de cigarrillos ya estaban bajando en todo EE.UU.). Necesitamos un **contrafactual**: ¿qué habría pasado en California *sin* la Proposición 99?

## 1. Marco Teórico: ¿Qué es el Control Sintético?

### La idea intuitiva

El método de Control Sintético construye una **versión artificial de California** (un "California sintético") combinando datos de otros estados que *no* implementaron la política. La combinación se elige para que el California sintético se parezca lo más posible al California real **antes** de la intervención.

### Formulación matemática

Sea $Y_{1t}^I$ el resultado observado para la unidad tratada (California) en el periodo $t$, y $Y_{1t}^N$ el resultado potencial sin intervención.

El **efecto del tratamiento** es:

$$\tau_{1t} = Y_{1t}^I - Y_{1t}^N \quad \text{para } t \geq T_0$$

donde $T_0$ es el periodo de intervención (1988).

El control sintético aproxima $Y_{1t}^N$ como una **combinación convexa** de las unidades del donor pool:

$$\hat{Y}_{1t}^N = \sum_{j=2}^{J+1} w_j \cdot Y_{jt}$$

sujeto a las restricciones:
- $w_j \geq 0$ para todo $j$ (pesos no negativos)
- $\sum_{j=2}^{J+1} w_j = 1$ (los pesos suman 1)

### ¿Cómo se eligen los pesos?

Los pesos $w^*$ minimizan la distancia entre California y el control sintético en el **periodo pre-intervención**:

$$w^* = \arg\min_w \sum_{t=1}^{T_0} \left( Y_{1t} - \sum_{j=2}^{J+1} w_j Y_{jt} \right)^2$$

### Ventajas sobre otros métodos

- **vs. Diff-in-Diff:** No requiere tendencias paralelas (supuesto fuerte y a menudo violado).
- **vs. Matching:** Usa combinaciones ponderadas, no solo una unidad de comparación.
- **Transparencia:** Los pesos revelan exactamente qué estados contribuyen al contrafactual.
- **Pruebas de placebo:** Permiten inferencia sin distribuciones paramétricas.

## 2. Carga de datos y librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficos
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

In [ ]:
# Cargar datos
df_raw = pd.read_csv('smoking_data.csv')

print(f'Dimensiones: {df_raw.shape}')
print(f'Estados: {df_raw["state"].nunique()}')
print(f'Periodo: {int(df_raw["year"].min())} - {int(df_raw["year"].max())}')

df_raw.head()

### Descripción de las variables

| Variable | Descripción |
|----------|-------------|
| `state` | Nombre del estado (unidad de tratamiento) |
| `year` | Año de observación |
| `cigsale` | Ventas per cápita de cigarrillos (en paquetes) — **variable de resultado** |
| `lnincome` | Log del ingreso per cápita |
| `beer` | Consumo per cápita de cerveza |
| `age15to24` | Proporción de la población entre 15 y 24 años |
| `retprice` | Precio minorista de cigarrillos |

## 3. Preparación de datos

In [ ]:
# Pivotar: cada fila = un estado, cada columna = un año
# Usamos solo 'cigsale' como variable de resultado
df = df_raw.pivot(index='state', columns='year', values='cigsale')

# Convertir nombres de columnas a enteros
df.columns = df.columns.astype(int)

print(f'Matriz de resultados: {df.shape[0]} estados × {df.shape[1]} años')
df.head()

## 4. Análisis Descriptivo

Antes de aplicar control sintético, visualicemos la tendencia de ventas de cigarrillos en California vs. el promedio de los demás estados.

In [ ]:
# Separar California del resto
california = df.loc['California']
otros_estados = df.drop('California').mean()

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(california.index, california.values, 'b-', linewidth=2.5, label='California')
ax.plot(otros_estados.index, otros_estados.values, 'r--', linewidth=2, label='Promedio otros estados')
ax.axvline(x=1988, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
ax.text(1988.5, california.max() * 0.97, 'Proposición 99\n(1988)', fontsize=10, color='gray')

ax.set_xlabel('Año')
ax.set_ylabel('Ventas per cápita de cigarrillos (paquetes)')
ax.set_title('Ventas de Cigarrillos: California vs. Promedio Nacional')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**Observación:** Ambas series muestran una tendencia a la baja desde los 80s. Sin embargo, después de 1988 California parece divergir más rápido. El problema es que el **promedio simple** de otros estados no es un buen contrafactual — California era diferente desde el principio. Aquí es donde entra el control sintético.

## 5. Implementación del Control Sintético

### Construyendo el control sintético desde cero

Implementaremos el método paso a paso para entender la mecánica. La idea es encontrar los pesos $w^*$ que minimizan el error cuadrático en el periodo pre-tratamiento.

In [ ]:
# ── Parámetros ──
TREATMENT_YEAR = 1988
TREATED_STATE = 'California'

# Separar periodos
pre_cols = [c for c in df.columns if c <= TREATMENT_YEAR]
post_cols = [c for c in df.columns if c > TREATMENT_YEAR]

print(f'Periodo pre-intervención: {pre_cols[0]}–{pre_cols[-1]} ({len(pre_cols)} años)')
print(f'Periodo post-intervención: {post_cols[0]}–{post_cols[-1]} ({len(post_cols)} años)')

In [ ]:
# ── Matrices de datos ──

# Vector de resultados de California (tratado)
Y1_pre  = df.loc[TREATED_STATE, pre_cols].values   # Pre-intervención
Y1_post = df.loc[TREATED_STATE, post_cols].values   # Post-intervención
Y1_all  = df.loc[TREATED_STATE].values              # Todo el periodo

# Matriz de donantes (todos los demás estados)
donors  = df.drop(TREATED_STATE)
Y0_pre  = donors[pre_cols].values     # J estados × T_pre periodos
Y0_post = donors[post_cols].values
Y0_all  = donors.values

donor_names = donors.index.tolist()

print(f'Unidad tratada: {TREATED_STATE}')
print(f'Pool de donantes: {len(donor_names)} estados')
print(f'Dimensión Y0_pre (donantes × periodos pre): {Y0_pre.shape}')

### La función objetivo

Buscamos los pesos $w$ que minimizan:

$$\text{RMSPE}_{\\text{pre}} = \sqrt{\\frac{1}{T_0} \sum_{t=1}^{T_0} \left( Y_{1t} - \sum_j w_j Y_{jt} \right)^2}$$

In [ ]:
def synthetic_control(Y1_pre, Y0_pre, Y0_all, X1=None, X0=None,
                      cov_weight=0.0, n_restarts=1):
    """
    Encuentra los pesos óptimos del control sintético.

    Parámetros:
    -----------
    Y1_pre     : array (T_pre,) — resultado de la unidad tratada, periodo pre
    Y0_pre     : array (J, T_pre) — resultados de donantes, periodo pre
    Y0_all     : array (J, T_all) — resultados de donantes, todo el periodo
    X1, X0     : arrays — predictores normalizados (opcional)
    cov_weight : float — peso relativo de covariables vs. outcome
    n_restarts : int — número de puntos iniciales aleatorios adicionales

    Retorna:
    --------
    weights : array (J,) — pesos óptimos
    Y_synth : array (T_all,) — serie del control sintético
    """
    J = Y0_pre.shape[0]

    def objective(w):
        out_loss = np.sum((Y1_pre - w @ Y0_pre) ** 2)
        if X1 is not None and X0 is not None and cov_weight > 0:
            cov_loss = np.sum((X1 - w @ X0) ** 2)
            return out_loss + cov_weight * cov_loss
        return out_loss

    constraints = {"type": "eq", "fun": lambda w: np.sum(w) - 1}
    bounds = [(0, 1)] * J

    # Múltiples puntos iniciales para evitar mínimos locales
    best_w, best_val = None, np.inf
    starts = [np.ones(J) / J]
    if n_restarts > 1:
        starts += [np.random.dirichlet(np.ones(J)) for _ in range(n_restarts - 1)]

    for w0 in starts:
        result = minimize(
            objective, w0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 2000, "ftol": 1e-14}
        )
        if result.fun < best_val:
            best_val = result.fun
            best_w = result.x

    Y_synth = best_w @ Y0_all
    return best_w, Y_synth

In [ ]:
# ── Construir predictores (covariables) siguiendo Abadie (2010) ──
covs = ["lnincome", "beer", "age15to24", "retprice"]
pre_data = df_raw[df_raw["year"] <= TREATMENT_YEAR]
cov_avg = pre_data.groupby("state")[covs].mean()

# Cigsale en años clave (como en el paper original)
specific_years = [1975, 1980, 1988]
cig_spec = df[specific_years]

# Combinar y normalizar
X = pd.concat([cov_avg, cig_spec], axis=1).fillna(cov_avg.mean())
X_norm = (X - X.min()) / (X.max() - X.min() + 1e-10)

X1 = X_norm.loc[TREATED_STATE].values
X0 = X_norm.drop(TREATED_STATE).values

# Ajustar con covariables y múltiples reinicios
np.random.seed(42)
weights, Y_synth = synthetic_control(
    Y1_pre, Y0_pre, Y0_all,
    X1=X1, X0=X0, cov_weight=5.0, n_restarts=10
)

rmspe_pre = np.sqrt(np.mean((Y1_pre - weights @ Y0_pre)**2))
print("✓ Control sintético ajustado exitosamente")
print(f"  RMSPE pre-intervención: {rmspe_pre:.2f} paquetes")
print(f"  Predictores usados: {covs} + cigsale en {specific_years}")

## 6. Análisis de Pesos

Una de las ventajas clave del control sintético es la **transparencia**: podemos ver exactamente qué estados contribuyen al contrafactual y con qué proporción.

In [ ]:
# Mostrar pesos significativos (> 1%)
weight_df = pd.DataFrame({
    'Estado': donor_names,
    'Peso': weights
}).sort_values('Peso', ascending=False)

weight_df_sig = weight_df[weight_df['Peso'] > 0.01].copy()
weight_df_sig['Peso_pct'] = (weight_df_sig['Peso'] * 100).round(1)

print('═' * 50)
print('PESOS DEL CONTROL SINTÉTICO')
print('═' * 50)
print(f'{"Estado":<20} {"Peso":>8} {"%":>8}')
print('─' * 50)
for _, row in weight_df_sig.iterrows():
    print(f'{row["Estado"]:<20} {row["Peso"]:>8.4f} {row["Peso_pct"]:>7.1f}%')
print('─' * 50)
print(f'{"Suma total":<20} {weights.sum():>8.4f} {weights.sum()*100:>7.1f}%')
print(f'\nEstados con peso > 0: {(weights > 0.01).sum()} de {len(weights)}')

In [ ]:
# Visualización de pesos
fig, ax = plt.subplots(figsize=(10, 5))

sig_mask = weights > 0.01
sig_names = [donor_names[i] for i in range(len(donor_names)) if sig_mask[i]]
sig_weights = weights[sig_mask]

# Ordenar de mayor a menor
sort_idx = np.argsort(sig_weights)[::-1]
sig_names = [sig_names[i] for i in sort_idx]
sig_weights = sig_weights[sort_idx]

colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(sig_names)))
bars = ax.barh(range(len(sig_names)), sig_weights, color=colors)

ax.set_yticks(range(len(sig_names)))
ax.set_yticklabels(sig_names)
ax.set_xlabel('Peso en el Control Sintético')
ax.set_title('Composición del California Sintético')
ax.invert_yaxis()

# Etiquetas de porcentaje
for bar, w in zip(bars, sig_weights):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{w*100:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.show()

**Interpretación de los pesos:** El control sintético de California es una combinación específica de estados que, en conjunto, reproducen la trayectoria pre-intervención de California. Nótese que la mayoría de los estados reciben peso cero — solo unos pocos son relevantes.

> **Nota conceptual:** La restricción de pesos no negativos que sumen 1 garantiza que el control sintético es una *interpolación* del donor pool, nunca una *extrapolación*.

## 7. Resultados Principales

In [ ]:
# ── California Real vs. California Sintético ──
years = df.columns.values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6),
                                gridspec_kw={"width_ratios": [2, 1]})

# Panel izquierdo: serie completa
ax1.plot(years, Y1_all, "b-", linewidth=2.5, label="California (observado)")
ax1.plot(years, Y_synth, "r--", linewidth=2, label="California sintético")
ax1.axvline(x=1988, color="black", linestyle=":", linewidth=1.2, alpha=0.6)
ax1.annotate("Proposición 99" + "\n" + "(1988)",
             xy=(1988, Y1_all.max() * 0.85),
             fontsize=10, ha="center", color="black",
             bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.5))
ax1.set_xlabel("Año")
ax1.set_ylabel("Ventas per cápita de cigarrillos (paquetes)")
ax1.set_title("California Real vs. California Sintético")
ax1.legend(frameon=False, loc="upper right")

# Panel derecho: zoom en periodo pre-intervención
pre_mask = years <= 1988
ax2.plot(years[pre_mask], Y1_all[pre_mask], "b-", linewidth=2.5, label="Observado")
ax2.plot(years[pre_mask], Y_synth[pre_mask], "r--", linewidth=2, label="Sintético")
ax2.set_xlabel("Año")
ax2.set_title("Zoom: Ajuste Pre-Intervención")
ax2.legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

# Métricas de ajuste pre-intervención
pre_diff = Y1_all[pre_mask] - Y_synth[pre_mask]
print(f"Ajuste pre-intervención (1970-1988):")
print(f"  RMSPE: {np.sqrt(np.mean(pre_diff**2)):.2f} paquetes")
print(f"  Error absoluto medio: {np.mean(np.abs(pre_diff)):.2f} paquetes")
print(f"  Error máximo: {np.max(np.abs(pre_diff)):.2f} paquetes")

**Lectura de la gráfica:** Antes de 1988 ambas líneas van casi juntas — el control sintético replica bien la trayectoria real. Después de 1988, la línea azul (observada) cae mucho más rápido que la roja (sintética). Esa brecha creciente es el **efecto causal estimado** de la Proposición 99.

Ahora agreguemos el promedio de otros estados para comparar qué tan superior es el control sintético como contrafactual:

In [ ]:
# ── Gráfica principal: California Real vs. Sintético ──
years = df.columns.values

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(years, Y1_all, 'b-', linewidth=2.5, label='California (observado)')
ax.plot(years, Y_synth, 'r--', linewidth=2, label='California sintético')
ax.plot(otros_estados.index, otros_estados.values,
        color='gray', alpha=0.4, linewidth=1, linestyle='-.',
        label='Promedio otros estados')

ax.axvline(x=1988, color='black', linestyle=':', linewidth=1.2, alpha=0.6)
ax.annotate('Proposición 99\n(1988)',
            xy=(1988, Y1_all.max() * 0.85),
            fontsize=10, ha='center', color='black',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='wheat', alpha=0.5))

# Sombrear el efecto post-intervención
post_mask = years > 1988
ax.fill_between(years[post_mask], Y1_all[post_mask], Y_synth[post_mask],
                alpha=0.15, color='red', label='Efecto estimado')

ax.set_xlabel('Año')
ax.set_ylabel('Ventas per cápita de cigarrillos (paquetes)')
ax.set_title('Control Sintético: Efecto de la Proposición 99')
ax.legend(frameon=False, loc='upper right')
plt.tight_layout()
plt.show()

**Interpretación clave:**
- **Antes de 1988:** El control sintético sigue de cerca a California real → buen ajuste pre-intervención.
- **Después de 1988:** Las series divergen significativamente → evidencia del efecto de la Proposición 99.
- El área sombreada representa la **reducción en ventas atribuible a la política**.
- El promedio simple (línea gris) no se ajusta tan bien como el control sintético en el periodo pre-intervención.

## 8. Efecto Estimado (Gap)

In [ ]:
# ── Gráfica del efecto (gap) ──
gap = Y1_all - Y_synth

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(years, gap, 'b-', linewidth=2)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.axvline(x=1988, color='gray', linestyle=':', linewidth=1.2, alpha=0.7)
ax.fill_between(years, gap, 0, where=(years > 1988), alpha=0.2, color='red')

ax.set_xlabel('Año')
ax.set_ylabel('Diferencia (Observado − Sintético)')
ax.set_title('Gap: Efecto de la Proposición 99 sobre Ventas de Cigarrillos')
ax.text(1989, gap.min() * 0.5, 'Proposición 99', fontsize=10, color='gray')

plt.tight_layout()
plt.show()

In [ ]:
# Efecto estimado al final del periodo
idx_2000 = list(years).index(2000)
effect_2000 = gap[idx_2000]

print("=" * 50)
print("EFECTO ESTIMADO (ano 2000):")
print(f"  California real:      {Y1_all[idx_2000]:.1f} paquetes per capita")
print(f"  California sintetico: {Y_synth[idx_2000]:.1f} paquetes per capita")
print(f"  Efecto (gap):         {effect_2000:.1f} paquetes per capita")
print(f"  Reduccion porcentual: {abs(effect_2000)/Y_synth[idx_2000]*100:.1f}%")
print("=" * 50)

**Resultado:** Para el año 2000, las ventas de cigarrillos en California fueron aproximadamente **26-30 paquetes menos per cápita** de lo que habrían sido sin la Proposición 99. Esto representa una reducción sustancial atribuible a la política.

## 9. Pruebas de Placebo (Inferencia)

### ¿Cómo sabemos que el efecto es "real"?

En el marco de control sintético, **no usamos p-values tradicionales**. En su lugar, aplicamos **pruebas de placebo in-space**: repetimos el análisis tratando a *cada estado* como si fuera la unidad tratada.

**Lógica:** Si el efecto que encontramos para California es inusualmente grande comparado con los "efectos" estimados para estados que *no* recibieron tratamiento, entonces tenemos evidencia de un efecto real.

### Procedimiento:
1. Para cada estado $j$ en el donor pool, estimamos un control sintético como si $j$ fuera el estado tratado.
2. Calculamos el gap (diferencia) para cada estado.
3. Comparamos el gap de California contra la distribución de gaps placebo.

In [ ]:
# ── Pruebas de Placebo ──
print("Ejecutando pruebas de placebo para todos los estados...")
print("(Esto puede tomar unos segundos)" + "\n")

all_states = df.index.tolist()
placebo_gaps = {}

for state in all_states:
    Y1_pre_p = df.loc[state, pre_cols].values
    Y1_all_p = df.loc[state].values

    donors_p = df.drop(state)
    Y0_pre_p = donors_p[pre_cols].values
    Y0_all_p = donors_p.values

    try:
        # n_restarts=1 para velocidad en los placebos
        w_p, Y_synth_p = synthetic_control(Y1_pre_p, Y0_pre_p, Y0_all_p,
                                            n_restarts=1)
        gap_p = Y1_all_p - Y_synth_p
        rmspe_pre = np.sqrt(np.mean((Y1_pre_p - w_p @ Y0_pre_p)**2))
        placebo_gaps[state] = {"gap": gap_p, "rmspe_pre": rmspe_pre}
    except:
        pass

print(f"✓ Completado: {len(placebo_gaps)} de {len(all_states)} estados")

In [ ]:
# ── Gráfica de placebos (todos los estados) ──
fig, ax = plt.subplots(figsize=(12, 6))

# Filtrar estados con mal ajuste pre-intervención
# (RMSPE pre > X veces el de California)
ca_rmspe = placebo_gaps['California']['rmspe_pre']
threshold = 5  # veces el RMSPE de California

for state, data in placebo_gaps.items():
    if state == 'California':
        continue
    if data['rmspe_pre'] <= ca_rmspe * threshold:
        ax.plot(years, data['gap'], color='lightgray', linewidth=0.8, alpha=0.6)

# California encima de todo
ax.plot(years, placebo_gaps['California']['gap'],
        'b-', linewidth=2.5, label='California')

ax.axhline(y=0, color='black', linewidth=0.8)
ax.axvline(x=1988, color='gray', linestyle=':', linewidth=1.2)
ax.text(1988.5, ax.get_ylim()[1] * 0.85, 'Proposición 99',
        fontsize=10, color='gray')

ax.set_xlabel('Año')
ax.set_ylabel('Gap (Observado − Sintético)')
ax.set_title(f'Pruebas de Placebo In-Space (estados con RMSPE pre ≤ {threshold}× California)')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**Interpretación:** Cada línea gris es el "efecto placebo" de un estado que no recibió tratamiento. Si el efecto de California (línea azul) es mucho mayor en magnitud que los efectos placebo, tenemos evidencia de que la Proposición 99 tuvo un efecto real.

> **Filtro RMSPE:** Excluimos estados con un ajuste pre-intervención muy pobre (RMSPE alto), ya que para esos estados el control sintético no es informativo.

### Ratio RMSPE post/pre

Una forma más rigurosa de evaluar la significancia es calcular el **ratio RMSPE post/pre** para cada estado:

$$\\text{Ratio}_j = \\frac{\\text{RMSPE}_{\\text{post}, j}}{\\text{RMSPE}_{\\text{pre}, j}}$$

El **p-value** se calcula como la fracción de estados con un ratio ≥ al de California.

In [ ]:
# ── Calcular ratios RMSPE post/pre ──
ratios = {}

for state, data in placebo_gaps.items():
    gap_s = data['gap']
    pre_idx = [i for i, y in enumerate(years) if y <= TREATMENT_YEAR]
    post_idx = [i for i, y in enumerate(years) if y > TREATMENT_YEAR]

    rmspe_pre_s = np.sqrt(np.mean(gap_s[pre_idx]**2))
    rmspe_post_s = np.sqrt(np.mean(gap_s[post_idx]**2))

    if rmspe_pre_s > 0:
        ratios[state] = rmspe_post_s / rmspe_pre_s

# Ordenar por ratio
ratio_df = pd.DataFrame({
    'Estado': list(ratios.keys()),
    'Ratio': list(ratios.values())
}).sort_values('Ratio', ascending=False).reset_index(drop=True)

# p-value de California
ca_ratio = ratios['California']
ca_rank = (ratio_df['Ratio'] >= ca_ratio).sum()
p_value = ca_rank / len(ratio_df)

print('═' * 55)
print('PRUEBA DE SIGNIFICANCIA: Ratio RMSPE post/pre')
print('═' * 55)
print(f'  Ratio de California: {ca_ratio:.2f}')
print(f'  Ranking: {ca_rank} de {len(ratio_df)}')
print(f'  p-value: {p_value:.3f} ({p_value*100:.1f}%)')
print('═' * 55)
print(f'\nTop 10 estados por ratio RMSPE:')
print(ratio_df.head(10).to_string(index=False))

In [ ]:
# ── Grafica de ratios ──
fig, ax = plt.subplots(figsize=(12, 8))

colors = ["#2171b5" if s == "California" else "#bdd7e7"
          for s in ratio_df["Estado"]]

bars = ax.barh(range(len(ratio_df)), ratio_df["Ratio"].values, color=colors)
ax.set_yticks(range(len(ratio_df)))
ax.set_yticklabels(ratio_df["Estado"].values, fontsize=8)
ax.set_xlabel("Ratio RMSPE (post/pre)")
ax.set_title("Distribucion de Ratios RMSPE: Prueba de Placebo")
ax.invert_yaxis()

plt.tight_layout()
plt.show()

**Resultado de la prueba de placebo:** Si California tiene el ratio mas alto (o uno de los mas altos), su p-value sera bajo, indicando que el efecto observado es **estadisticamente inusual** y probablemente atribuible a la Proposicion 99.

> Un p-value de ~1/39 = 0.026 seria equivalente a significancia al 5%.

## 10. Placebo Temporal (In-Time)

Otra prueba de robustez: ¿qué pasa si aplicamos el control sintético usando un **año de intervención falso** antes de 1988? Si encontramos un "efecto" grande, significaría que nuestro método no es confiable.

In [ ]:
# ── Placebo temporal: año de intervención falso en 1983 ──
PLACEBO_YEAR = 1983

pre_cols_placebo = [c for c in df.columns if c <= PLACEBO_YEAR]
cols_until_1988 = [c for c in df.columns if c <= TREATMENT_YEAR]

Y1_pre_pt = df.loc[TREATED_STATE, pre_cols_placebo].values
donors_pt = df.drop(TREATED_STATE)
Y0_pre_pt = donors_pt[pre_cols_placebo].values
Y0_until_1988 = donors_pt[cols_until_1988].values

w_pt, Y_synth_pt = synthetic_control(Y1_pre_pt, Y0_pre_pt, Y0_until_1988,
                                      n_restarts=1)

Y1_until_1988 = df.loc[TREATED_STATE, cols_until_1988].values
gap_pt = Y1_until_1988 - Y_synth_pt

# Gráfica
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(cols_until_1988, gap_pt, "b-", linewidth=2)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.axvline(x=PLACEBO_YEAR, color="red", linestyle=":", linewidth=1.2)
ax.text(PLACEBO_YEAR + 0.3, gap_pt.max() * 0.8,
        f"Placebo ({PLACEBO_YEAR})", fontsize=10, color="red")
ax.axvline(x=TREATMENT_YEAR, color="gray", linestyle=":", linewidth=1)
ax.text(TREATMENT_YEAR + 0.3, gap_pt.min() * 0.5,
        f"Prop 99 real ({TREATMENT_YEAR})", fontsize=10, color="gray")

ax.set_xlabel("Año")
ax.set_ylabel("Gap (Observado - Sintético)")
ax.set_title(f"Placebo Temporal: Intervención Ficticia en {PLACEBO_YEAR}")
plt.tight_layout()
plt.show()

pre_len = len(pre_cols_placebo)
print(f"Gap promedio post-placebo ({PLACEBO_YEAR}-{TREATMENT_YEAR}): {gap_pt[pre_len:].mean():.2f}")
print("→ Un gap cercano a 0 indica que el método es válido (no hay efecto espurio).")

**Interpretación:** Si el gap se mantiene cercano a cero entre 1983 y 1988, confirma que el control sintético no está "inventando" efectos donde no los hay. Esto refuerza la credibilidad del efecto encontrado después de 1988.

## 11. Resumen y Conclusiones

### Hallazgos principales

1. **El control sintético reproduce bien la trayectoria pre-intervención** de California, lo que valida el método.
2. **Efecto sustancial de la Proposición 99:** Las ventas per cápita de cigarrillos en California cayeron significativamente más de lo esperado (~26-30 paquetes menos para el año 2000).
3. **Las pruebas de placebo confirman la significancia** del efecto: el gap de California es inusualmente grande comparado con los efectos placebo de otros estados.
4. **El placebo temporal** descarta que el método genere efectos espurios.

### Limitaciones del método

- **Interpolación solamente:** Si la unidad tratada está fuera del "rango" del donor pool, el SC no puede aproximarla bien.
- **Sensibilidad al donor pool:** Excluir/incluir estados puede cambiar los resultados.
- **Solo una unidad tratada:** El método clásico analiza una sola unidad. Extensiones como el SC generalizado abordan múltiples unidades.
- **No captura spillovers:** Si la política de California afectó a estados vecinos, el SC estaría sesgado.

### Referencias

- Abadie, A., Diamond, A., & Hainmueller, J. (2010). *Synthetic Control Methods for Comparative Case Studies*. JASA, 105(490).
- Abadie, A., Diamond, A., & Hainmueller, J. (2015). *Comparative Politics and the Synthetic Control Method*. AJPS, 59(2).
- Abadie, A. (2021). *Using Synthetic Controls: Feasibility, Data Requirements, and Methodological Aspects*. JEL, 59(2).